# Tasks remaining to do:

1. Encode the filename by using the encoder from wisio "https://github.com/izzet/wisio/blob/main/tools/recorder2parquet.cpp#L727"

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import json
import os
import yaml
from pathlib import Path
from dask.distributed import Client
import dask.dataframe as dd
import networkx as nx
import sys
import pandas as pd
import re
import glob


import ipycytoscape
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import intervals
import pygtrie
import seaborn as sns

/usr/workspace/pandey2/DFtracer/envdft/lib/python3.9/site-packages/dask/dataframe/__init__.py:49: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [5]:
# App Name
app_name = "mummi" # resnet cosmoflow unet3d mummi deepspeed dlio_resnet dlio_cosmoflow
# app_name = "montage"
# app_name = "deepspeed2"
# app_name = "deepspeed1"
# app_name = "deepspeed"
# app_name = "1000genome"
# cp_dir = "/p/lustre2/pandey2/cp_dir"


In [10]:

use_local=False
if not use_local:
    with open(f'/g/g91/pandey2/.dftracer/configuration.yaml', 'r') as file:
    # 1 with open(f'/g/g91/pandey2/.dlio_profiler/configuration.yaml', 'r') as file:
        dlp_yaml = yaml.safe_load(file)
        app_root = dlp_yaml["app"]
else:
    app_root = str(Path(os.getcwd()).parent.parent)
sys.path.insert(0, app_root)


import dfanalyzer
print(dfanalyzer.__file__)
from dfanalyzer.main import DFAnalyzer,get_dft_configuration,update_dft_configuration,setup_logging,setup_dask_cluster, reset_dask_cluster, get_dft_configuration
from dfanalyzer.graph_visualization.plots import GrepIOPlots
from dfanalyzer.graph_visualization.cytoscape import GraphFunctions, CytoGraph
from dfanalyzer.graph import GrepIO 

if not use_local:
    dask_run_dir = os.path.join(app_root, "dfanalyzer", "dask", "run_dir")
    with open (os.path.join(dask_run_dir, f"scheduler_{os.getenv('USER')}.json"), "r") as f:
        dask_scheduler = json.load(f)["address"]
else:
    dask_scheduler = None


def get_conditions_cosmoflow(json_object):
    app_io_cond = "TFReader.parse_image" in json_object["name"] # Cosmoflow
    compute_cond = "compute" in json_object["name"] # Cosmoflow
    io_cond = "POSIX" == json_object["cat"] # Cosmoflow
    return app_io_cond, compute_cond, io_cond

def get_conditions_resnet(json_object):
    app_io_cond = "IO" == json_object["cat"] # Resnet50
    compute_cond = "cpu" in json_object["name"] or "compute" in json_object["cat"] # Resnet50
    io_cond = "POSIX" == json_object["cat"] # Cosmoflow
    return app_io_cond, compute_cond, io_cond

def get_conditions_unet3d(json_object):
    app_io_cond = "NPZReader.read_index" in json_object["name"] # Unet3d
    compute_cond = "compute" in json_object["name"] # Unet3d
    io_cond = "POSIX" == json_object["cat"] # Cosmoflow
    return app_io_cond, compute_cond, io_cond

def get_conditions_dlio_resnet(json_object):
    app_io_cond = "read_index" in json_object["name"] # Unet3d
    compute_cond = "compute" in json_object["name"] # Unet3d
    io_cond = "POSIX" == json_object["cat"] # Cosmoflow
    return app_io_cond, compute_cond, io_cond


condition_fn = None #

if app_name == "mummi":
    filename = "/usr/workspace/iopp/graph-io/dlp_logs/mummi-32-node/*pfw.gz"

elif app_name == "montage":
    filename ="/usr/workspace/iopp/graph-io/dlp_logs/montage_16_48ppn/montage*.pfw"

elif app_name == "deepspeed":
    # filename = "/usr/workspace/iopp/graph-io/dlp_logs/deepspeed_8_4ppn/*.pfw.gz"
    filename = "/usr/workspace/iopp/dlp_traces/deepspeed_8_4ppn/*.pfw.gz"
    
elif app_name == "deepspeed1": #deepspeed low interference
    filename = "/usr/workspace/iopp/graph-io/dlp_logs/deepspeed_2_node_low_interference/*.pfw.gz"

elif app_name == "deepspeed2": # deepspeed high interference
    filename = "/usr/workspace/iopp/graph-io/dlp_logs/deepspeed_2_node_high_interference/*.pfw.gz"

elif app_name == "1000genome":
    filename = "/usr/workspace/iopp/graph-io/dlp_logs/1000genome/*.gz"
    
elif app_name == "cosmoflow":#empty traces do not run yet
    filename = "/usr/WS2/iopp/kogiou1/dlio_benchmark/hydra_log/cosmoflow/2023-10-31-10-27-24/.trace*.pfw.gz"
    condition_fn = get_conditions_cosmoflow

elif app_name == "node32":#empty traces do not run yet
    filename = "/usr/workspace/iopp/dlp_traces/node_32_ppn_4/.trace*.pfw.gz"

elif app_name == "resnet":
    filename = "/usr/workspace/iopp/dlp_traces/resnet_50_1node_4ppn/*.pfw.gz"
    condition_fn = get_conditions_dlio_resnet

else:
    raise Exception("Unknown App name")


# Configuration 4 update log file dlp -> df
conf = update_dft_configuration(dask_scheduler=dask_scheduler, verbose=True, 
                                log_file=f"./dft_{os.getenv('USER')}.log", rebuild_index=False, time_approximate=False, 
                                host_pattern=r'lassen(\d+)', time_granularity=30e6, skip_hostname=True, conditions=condition_fn)
conf = get_dft_configuration()


# Setup
setup_logging()
setup_dask_cluster()
reset_dask_cluster()

[INFO] [18:18:23] Initialized Client with 0 workers and link http://134.9.71.20:8787/status [/usr/workspace/pandey2/DFtracer1/dftracer/dfanalyzer/main.py:672]
[INFO] [18:18:23] Restarting all workers [/usr/workspace/pandey2/DFtracer1/dftracer/dfanalyzer/main.py:664]


/usr/workspace/pandey2/DFtracer1/dftracer/dfanalyzer/__init__.py


In [15]:
# This cell is for mount point detection functions

def find_mount_point(path,trie):
    mount_point = trie.longest_prefix(path)
    if mount_point:
        return mount_point.key
    return "/"

def all_mount_points():
    with open("/proc/mounts", "r") as file:
        mount_points = [line.split()[1] for line in file]
    with open("/usr/workspace/pandey2/lassen_mounts", "r") as file:
        mount_p = [line.split()[1] for line in file]
    return mount_points+mount_p

mount_points = all_mount_points()
trie = pygtrie.StringTrie(zip(mount_points, [True] * len(mount_points)))

def mummi_cols_function(json_object, current_dict, time_approximate,condition_fn,load_data):
    d = {}
    def find_mount_point(path,trie):
        mount_point = trie.longest_prefix(path)
        if mount_point:
            return mount_point.key
        return "/"
    
    if "args" in json_object:
        if "path" in json_object["args"]:
            d["path"] = str(json_object["args"]["path"])
        if "filename" in json_object["args"]:
            d["filename"] = str(json_object["args"]["filename"])   
            d["mount_point"] = find_mount_point(trie=load_data["mount_point"],path=str(json_object["args"]["filename"]))
        if "flags" in json_object["args"]:
            d['flags'] = int(json_object["args"]["flags"])
        if "mode" in json_object["args"]:
            d['mode'] = int(json_object["args"]["mode"])
            
    if "name" in json_object:
        if (json_object["name"] == "write"):
            d["prod"] = 1
            d["cons"] = 0
        else:
            d["prod"] = 0
            d["cons"] = 1      
    return d


def montage_cols_function(json_object, current_dict, time_approximate,condition_fn,load_data):
    d = {}
    def find_mount_point(path,trie):
        mount_point = trie.longest_prefix(path)
        if mount_point:
            return mount_point.key
        return "/"
    
    if "args" in json_object:
        if "path" in json_object["args"]:
            d["path"] = str(json_object["args"]["path"])
        if "fname" in json_object["args"]:
            d["filename"] = str(json_object["args"]["fname"])   
            d["mount_point"] = find_mount_point(trie=load_data["mount_point"],path=str(json_object["args"]["fname"]))
        if "flags" in json_object["args"]:
            d['flags'] = int(json_object["args"]["flags"])
        if "mode" in json_object["args"]:
            d['mode'] = int(json_object["args"]["mode"])

    if "name" in json_object:
        if (json_object["name"] == "fwrite"):
            d["prod"] = 1
            d["cons"] = 0
        else:
            d["prod"] = 0
            d["cons"] = 1       
    return d

def deepspeed_cols_function(json_object, current_dict, time_approximate,condition_fn,load_data):
    d = {}
    def find_mount_point(path,trie):
        mount_point = trie.longest_prefix(path)
        if mount_point:
            return mount_point.key
        return "/"
    
    if "args" in json_object:
        if "fname" in json_object["args"]:
            d["mount_point"] = find_mount_point(trie=load_data["mount_point"],path=str(json_object["args"]["fname"]))
    
    if "name" in json_object:
        if (json_object["name"] == "write"):
            d["prod"] = 1
            d["cons"] = 0
        else:
            d["prod"] = 0
            d["cons"] = 1      
    return d

def genome_cols_function(json_object, current_dict, time_approximate,condition_fn,load_data):
    d = {}
    def find_mount_point(path,trie):
        mount_point = trie.longest_prefix(path)
        if mount_point:
            return mount_point.key
        return "/"
    
    if "args" in json_object:
        if "fname" in json_object["args"]:
            d["mount_point"] = find_mount_point(trie=load_data["mount_point"],path=str(json_object["args"]["fname"]))
    
    if "name" in json_object:
        if (json_object["name"] == "write"):
            d["prod"] = 1
            d["cons"] = 0
        else:
            d["prod"] = 0
            d["cons"] = 1      
    return d

load_cols_mummi = {'path':"string[pyarrow]", 'filename':"string[pyarrow]",'mount_point':"string[pyarrow]", 'flags':"uint64[pyarrow]", 'mode':"uint64[pyarrow]", 'prod':"uint16[pyarrow]", 'cons':"uint16[pyarrow]"}
load_cols_montage = {'path':"string[pyarrow]", 'filename':"string[pyarrow]",'mount_point':"string[pyarrow]", 'flags':"uint64[pyarrow]", 'mode':"uint64[pyarrow]", 'prod':"uint16[pyarrow]", 'cons':"uint16[pyarrow]"}
load_cols_deepspeed = {'mount_point':"string[pyarrow]",'prod':"uint16[pyarrow]", 'cons':"uint16[pyarrow]"}
load_cols_genome = {'mount_point':"string[pyarrow]",'prod':"uint16[pyarrow]", 'cons':"uint16[pyarrow]"}

In [16]:
# analyzer_mummi1 = DFAnalyzer(filename,load_fn=mummi_cols_function, load_cols=load_cols_mummi, load_data={"mount_point":trie})
#analyzer_deepspeed = DFAnalyzer(filename,load_fn=deepspeed_cols_function, load_cols=load_cols_deepspeed, load_data={"mount_point":trie})
analyzer_montage = DFAnalyzer(filename,load_fn=montage_cols_function, load_cols=load_cols_montage, load_data={"mount_point":trie})
# analyzer_deepspeed1 = DFAnalyzer(filename)
# analyzer_deepspeed = DFAnalyzer(filename,load_fn=deepspeed_cols_function, load_cols=load_cols_deepspeed, load_data={"mount_point":trie})
#analyzer_1000genome = DFAnalyzer(filename,load_fn=genome_cols_function, load_cols=load_cols_genome, load_data={"mount_point":trie})

[INFO] [13:00:12] Created index for 0 files [/usr/workspace/pandey2/DFtracer1/dftracer/dfanalyzer/main.py:370]
[INFO] [13:00:12] Total size of all files are <dask.bag.core.Item object at 0x1554ac699610> bytes [/usr/workspace/pandey2/DFtracer1/dftracer/dfanalyzer/main.py:372]
[INFO] [13:00:33] Loaded events [/usr/workspace/pandey2/DFtracer1/dftracer/dfanalyzer/main.py:430]
[INFO] [13:00:33] Loaded plots with slope threshold: 45 [/usr/workspace/pandey2/DFtracer1/dftracer/dfanalyzer/main.py:436]


In [17]:
analyzer_montage.events.query('prod == 1').compute()

,name,cat,pid,tid,ts,te,dur,tinterval,trange,hostname,...,total_time,filename,phase,size,path,mount_point,flags,mode,prod,cons
24,fwrite,STDIO,617137,1234274,232354197,232376321,22124,"[1717621614835805,1717621614857929]",7,corona237,...,(),3-mosaic.fits,0,<NA>,<NA>,/,<NA>,<NA>,1,0
25,fwrite,STDIO,617137,1234274,232376341,232376341,0,[1717621614857949],7,corona237,...,(),3-mosaic.fits,0,<NA>,<NA>,/,<NA>,<NA>,1,0
26,fwrite,STDIO,617137,1234274,232376348,232378176,1828,"[1717621614857956,1717621614859784]",7,corona237,...,(),3-mosaic.fits,0,<NA>,<NA>,/,<NA>,<NA>,1,0
27,fwrite,STDIO,617137,1234274,232378203,232394960,16757,"[1717621614859811,1717621614876568]",7,corona237,...,(),3-mosaic_area.fits,0,<NA>,<NA>,/,<NA>,<NA>,1,0
28,fwrite,STDIO,617137,1234274,232394971,232394971,0,[1717621614876579],7,corona237,...,(),3-mosaic_area.fits,0,<NA>,<NA>,/,<NA>,<NA>,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27883,fwrite,STDIO,984793,1969586,2304925,2304930,5,"[1717621384786533,1717621384786538]",0,corona267,...,(),/tmp/mv2-hwloc-f2QPQ6Hef-corona267-63170-whole...,0,<NA>,<NA>,/tmp,<NA>,<NA>,1,0
27884,fwrite,STDIO,984793,1969586,2304955,2304958,3,"[1717621384786563,1717621384786566]",0,corona267,...,(),/tmp/mv2-hwloc-f2QPQ6Hef-corona267-63170-whole...,0,<NA>,<NA>,/tmp,<NA>,<NA>,1,0
27885,fwrite,STDIO,984793,1969586,2304988,2304992,4,"[1717621384786596,1717621384786600]",0,corona267,...,(),/tmp/mv2-hwloc-f2QPQ6Hef-corona267-63170-whole...,0,<NA>,<NA>,/tmp,<NA>,<NA>,1,0
27886,fwrite,STDIO,984793,1969586,2305018,2305021,3,"[1717621384786626,1717621384786629]",0,corona267,...,(),/tmp/mv2-hwloc-f2QPQ6Hef-corona267-63170-whole...,0,<NA>,<NA>,/tmp,<NA>,<NA>,1,0


In [8]:
# from dfanalyzer.graph import DFWorkflowgraph
# # genome Graph
# dfworkflow_genome = DFWorkflowgraph(analyzer_1000genome.events, app_name = "1000genome", trace_path=filename)
# #perform reduction
# wf = dfworkflow_genome.create_workflow()
# # pid_map = dfworkflow_montage.get_pid_map()
# graph_df = dfworkflow_genome.create_graph_df(wf,pid_map={})
# temp_graph = graph_df.groupby(['src','dest'])['wt'].sum().reset_index()
# temp_graph.sort_values('wt')
# wf.compute()

,filename,pid,prod_pid,cons_pid,ts,prod_fid,cons_fid
0,/usr/workspace/iopp/kogiou1/workflows/pegasus/...,1055968,0,1,2550640420,1,13
1,/usr/workspace/iopp/kogiou1/workflows/pegasus/...,1055968,0,1,2550670153,1,13
2,/g/g92/kogiou1/.cache/matplotlib/fontlist-v390...,1055968,2,6,2551314166,36,524
3,/usr/workspace/iopp/kogiou1/workflows/pegasus/...,1055968,0,1,2554998533,1,7
4,/usr/workspace/iopp/kogiou1/workflows/pegasus/...,1055968,1,4,2555039177,1,7
...,...,...,...,...,...,...,...
11977594,./chr2n/chr2.NA21137,981916,1,13,1922887101486,21,297
11977595,./chr2n/chr2.NA21141,981916,1,13,1922887109653,21,293
11977596,./chr2n/chr2.NA21142,981916,1,13,1922887115884,21,289
11977597,./chr2n/chr2.NA21143,981916,1,13,1922887121290,21,281


In [9]:
# from dfanalyzer.graph import DFWorkflowgraph
# # Deepspeed Graph
# dfworkflow_deepspeed = DFWorkflowgraph(analyzer_deepspeed.events, app_name = "deepspeed", trace_path=filename)
# #perform reduction
# wf = dfworkflow_deepspeed.create_workflow()
# pid_map = dfworkflow_montage.get_pid_map()
# graph_df = dfworkflow_deepspeed.create_graph_df(wf,pid_map={})
# temp_graph = graph_df.groupby(['src','dest'])['wt'].sum().reset_index()
# temp_graph.sort_values('wt')
# wf.compute()

NameError: name 'dfworkflow_montage' is not defined

In [18]:
from dfanalyzer.graph import DFWorkflowgraph
# Montage Graph
dfworkflow_montage = DFWorkflowgraph(analyzer_montage.events, app_name = "montage", trace_path=filename)
#perform reduction
wf = dfworkflow_montage.create_workflow()
# pid_map = dfworkflow_montage.get_pid_map()
graph_df = dfworkflow_montage.create_graph_df(wf,pid_map={})
temp_graph = graph_df.groupby(['src','dest'])['wt'].sum().reset_index()
temp_graph.sort_values('wt')

,src,dest,wt
2,1-corrections.tbl,826708,1
32,2-mosaic.png,617149,1
51,3-mosaic.png,617154,1
39,3-corrections.tbl,826707,1
13,1-mosaic.png,617146,1
...,...,...,...
31,2-mosaic.png,617148,337101722
71,617154,3-mosaic.png,338300946
50,3-mosaic.png,617152,338384737
70,617153,mosaic-color.png,343614277


In [19]:
x = temp_graph.groupby(['src', 'dest'])['wt'].min()

In [23]:
x.reset_index().sort_values('wt').to_csv("montage.csv", index = False)

In [13]:
temp_graph.src.isin(temp_graph.dest).all()

np.True_

In [15]:
temp_graph.to_csv("montageVis1.csv", index=False)

In [10]:
fname = "/usr/workspace/iopp/graph-io/dlp_logs/deepspeed_2_node_low_interference/*.pfw.gz"

In [11]:
analyzer_deepspeed1 = DFAnalyzer("/usr/workspace/iopp/graph-io/dlp_logs/deepspeed_2_node_low_interference/*.pfw.gz", load_fn=deepspeed_cols_function, load_cols=load_cols_deepspeed, load_data={"mount_point":trie})

[INFO] [17:40:10] Created index for 16 files [/usr/workspace/pandey2/DFtracer1/dftracer/dfanalyzer/main.py:370]
[INFO] [17:40:10] Total size of all files are <dask.bag.core.Item object at 0x1554ac679ac0> bytes [/usr/workspace/pandey2/DFtracer1/dftracer/dfanalyzer/main.py:372]
[INFO] [17:40:12] Loading 8992 batches out of 16 files and has 147114386 lines overall [/usr/workspace/pandey2/DFtracer1/dftracer/dfanalyzer/main.py:385]
[INFO] [17:41:33] Loaded events [/usr/workspace/pandey2/DFtracer1/dftracer/dfanalyzer/main.py:430]
[INFO] [17:41:33] Loaded plots with slope threshold: 45 [/usr/workspace/pandey2/DFtracer1/dftracer/dfanalyzer/main.py:436]


In [12]:
from dfanalyzer.graph import DFWorkflowgraph
# Mummi Graph
# dfworkflow = DFWorkflowgraph(analyzer_mummi1.events, app_name = "mummi", trace_path=filename)
# #perform reduction
# wf = dfworkflow.create_workflow()
# pid_map = dfworkflow.get_pid_map()
# graph_df = dfworkflow.create_graph_df(wf,pid_map)
# temp_graph = graph_df.groupby(['src','dest'])['wt'].sum().reset_index()
# temp_graph.sort_values('wt')

In [13]:
dfworkflow_deepspeed1 = DFWorkflowgraph(analyzer_deepspeed1.events, app_name = "deepspeed1", trace_path=fname)
wf_deepspeed1 = dfworkflow_deepspeed1.create_workflow()

In [14]:
wf_deepspeed1.compute()

,filename,pid,prod_pid,cons_pid,ts,prod_fid,cons_fid
51,/l/ssd/haridev/scr/checkpoints/scr_megatron_de...,0,1,4,70549921,2,8
52,/l/ssd/haridev/scr/checkpoints/scr_megatron_de...,0,1,4,70550449,2,8
100,/l/ssd/haridev/scr/checkpoints/scr_megatron_de...,0,1,4,70632854,2,8
101,/l/ssd/haridev/scr/checkpoints/scr_megatron_de...,0,1,4,70633339,2,8
149,/l/ssd/haridev/scr/checkpoints/scr_megatron_de...,0,1,4,70712611,2,8
...,...,...,...,...,...,...,...
1994,/l/ssd/haridev/scr/checkpoints/scr_megatron_de...,8,1,3,1001551254,2,6
2042,/l/ssd/haridev/scr/checkpoints/scr_megatron_de...,8,2,6,1177204138,4,12
2045,/l/ssd/haridev/scr/checkpoints/scr_megatron_de...,8,1,3,1177441285,2,6
2046,/l/ssd/haridev/scr/checkpoints/scr_megatron_de...,8,2,6,1177560730,4,12


In [35]:
wf_deepspeed1.head()

In [15]:
deepspeed1_graph_data = dfworkflow_deepspeed1.create_graph_df(wf_deepspeed1,{})

In [16]:
deepspeed1_graph_data

,src,dest,wt
0,/l/ssd/haridev/scr/checkpoints/scr_megatron_de...,0,4
1,0,/l/ssd/haridev/scr/checkpoints/scr_megatron_de...,70549921
2,/l/ssd/haridev/scr/checkpoints/scr_megatron_de...,0,4
3,0,/l/ssd/haridev/scr/checkpoints/scr_megatron_de...,70550449
4,/l/ssd/haridev/scr/checkpoints/scr_megatron_de...,0,4
...,...,...,...
139,8,/l/ssd/haridev/scr/checkpoints/scr_megatron_de...,1177441285
140,/l/ssd/haridev/scr/checkpoints/scr_megatron_de...,8,6
141,8,/l/ssd/haridev/scr/checkpoints/scr_megatron_de...,1177560730
142,/l/ssd/haridev/scr/checkpoints/scr_megatron_de...,8,3


In [17]:
edgelist = deepspeed1_graph_data.groupby(['src','dest'])['wt'].min()

In [19]:
edgelist.reset_index()

,src,dest,wt
0,/l/ssd/haridev/scr/checkpoints/scr_megatron_de...,0,4
1,/l/ssd/haridev/scr/checkpoints/scr_megatron_de...,8,4
2,/l/ssd/haridev/scr/checkpoints/scr_megatron_de...,0,3
3,/l/ssd/haridev/scr/checkpoints/scr_megatron_de...,8,3
4,/l/ssd/haridev/scr/checkpoints/scr_megatron_de...,0,4
5,/l/ssd/haridev/scr/checkpoints/scr_megatron_de...,8,4
6,/l/ssd/haridev/scr/checkpoints/scr_megatron_de...,0,3
7,/l/ssd/haridev/scr/checkpoints/scr_megatron_de...,8,3
8,0,/l/ssd/haridev/scr/checkpoints/scr_megatron_de...,70550449
9,0,/l/ssd/haridev/scr/checkpoints/scr_megatron_de...,296730576


In [20]:
edgelist.reset_index().to_csv("dslow.csv",index = False)

In [9]:
#For deepspeed2
dfworkflow = DFWorkflowgraph(analyzer_deepspeed.events, app_name = "deepspeed", trace_path=filename)

In [10]:
prod_cons = analyzer_deepspeed.events.groupby('filename')['prod','cons'].sum()

In [11]:
prod_cons = prod_cons.query('prod > 0 and cons > 0').reset_index()

In [12]:
filelist = prod_cons.filename.unique().compute()

In [15]:
selected_events = analyzer_deepspeed.events[analyzer_deepspeed.events.filename.isin(filelist)]

In [17]:
selected_events_sum = selected_events.groupby(['filename', 'pid']).agg({'prod': 'sum', 'cons':'sum', 'ts':'min'}).reset_index()


In [18]:
selected_events_sum.head()

,filename,pid,prod,cons,ts
0,/p/lustre2/haridev/dlio/scr/checkpoints/scr_me...,0,1,3,62065109
1,/p/lustre2/haridev/dlio/scr/checkpoints/scr_me...,0,8,64,62071847
2,/p/lustre2/haridev/dlio/scr/checkpoints/scr_me...,0,31,399,62090408
3,/p/lustre2/haridev/dlio/scr/checkpoints/scr_me...,0,24,170,62204118
4,/p/lustre2/haridev/dlio/scr/checkpoints/scr_me...,0,2,8,62505785


In [27]:
prod_cons['prod'].max().compute()

np.int64(47)

In [28]:
selected_events_sum['prod'].max().compute()

np.int64(47)

In [19]:
merged = selected_events_sum.merge(prod_cons, on = ["filename"], how = 'left', suffixes = ['_pid','_fid'])

In [20]:
merged = selected_events_sum.merge(prod_cons, on = ["filename"], how = 'left', suffixes = ['_pid','_fid'])

,filename,pid,prod_pid,cons_pid,ts,prod_fid,cons_fid
0,/p/lustre2/haridev/dlio/scr/checkpoints/scr_me...,0,1,3,62065109,1,3
1,/p/lustre2/haridev/dlio/scr/checkpoints/scr_me...,0,8,64,62071847,8,64
2,/p/lustre2/haridev/dlio/scr/checkpoints/scr_me...,0,31,399,62090408,31,399
3,/p/lustre2/haridev/dlio/scr/checkpoints/scr_me...,0,24,170,62204118,24,170
4,/p/lustre2/haridev/dlio/scr/checkpoints/scr_me...,0,2,8,62505785,2,8


In [23]:
len(merged)

1483

In [21]:
final = merged.query('not (prod_pid == prod_fid and cons_pid == cons_fid)')

In [22]:
final.head()

,filename,pid,prod_pid,cons_pid,ts,prod_fid,cons_fid


In [30]:
df = pd.read_csv("/usr/workspace/pandey2/g_analyzer/MummiVis4.csv")
df['src'] = df['src'].astype(str)[5:]

In [31]:
df

,src,dest,wt
0,NaN,cganalysis,18
1,NaN,createsims,1908
2,NaN,mlserver,3424
3,NaN,14251,348
4,NaN,cganalysis,7
5,/p/gpfsx/iopp/mummi_demoroot_x_profile/sims-cg...,createsims,322
6,/p/gpfsx/iopp/mummi_demoroot_x_profile/sims-cg...,14251,15746
7,/p/gpfsx/iopp/mummi_demoroot_x_profile/sims-cg...,cganalysis,7
8,/p/gpfsx/iopp/mummi_demoroot_x_profile/sims-cg...,createsims,945
9,cganalysis,/p/gpfsx/iopp/mummi_demoroot_x_profile/sims-cg...,1


In [21]:
df

,src,dest,wt,trange
0,/p/gpfsx/iopp/mummi_demoroot_x_profile/ml/iter...,cganalysis,18,0
1,/p/gpfsx/iopp/mummi_demoroot_x_profile/ml/iter...,createsims,1908,1
2,/p/gpfsx/iopp/mummi_demoroot_x_profile/ml/iter...,mlserver,3424,2
3,/p/gpfsx/iopp/mummi_demoroot_x_profile/sims-cg...,14251,348,3
4,/p/gpfsx/iopp/mummi_demoroot_x_profile/sims-cg...,cganalysis,7,4
5,/p/gpfsx/iopp/mummi_demoroot_x_profile/sims-cg...,createsims,322,5
6,/p/gpfsx/iopp/mummi_demoroot_x_profile/sims-cg...,14251,15746,6
7,/p/gpfsx/iopp/mummi_demoroot_x_profile/sims-cg...,cganalysis,7,7
8,/p/gpfsx/iopp/mummi_demoroot_x_profile/sims-cg...,createsims,945,8
9,cganalysis,/p/gpfsx/iopp/mummi_demoroot_x_profile/sims-cg...,1,9


In [24]:
import pandas as pd
df = pd.read_csv("/usr/workspace/pandey2/g_analyzer/MummiVis4.csv")
df['trange'] = df.index
c = CytoGraph()
c.temporal_view(df)